# Training with Cleaned Dataset and EDL Type 2
A notebook for training an EfficientNet B3 model with Evidential Deep Learning (EDL Type 2) on a 5-class diabetic retinopathy dataset.
Dataset: `cleaned_dataset` with `train`, `val`, and `test` splits and 0-4 class folders.

## Execution Flow (Single Best Path Only)
1. Run Cells 1-6
2. Run Cell 7, then Cell 8 immediately after it
3. Run Cell 9 at the end

Cells 7 and 8 are intentionally adjacent so training and evaluation stay in one pass.

## 1. Import Core Libraries

In [1]:
!pip install torch torchvision timm efficientnet-pytorch scikit-learn pandas matplotlib seaborn opencv-python

import os
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import timm
from efficientnet_pytorch import EfficientNet
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=9bf1f215bb0e4bcbc5b173c76ab56dd10a56dc63cc0cac6b4df8342d8b41b70d
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet-pytorch


## 2. Define Hyperparameters and Device Configuration

In [2]:
import kagglehub

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
NUM_EPOCHS = 60
NUM_CLASSES = 5
PATIENCE = 15

# Keep notebook loaders Windows-safe; a small worker count is a better balance than the default 8.
NUM_WORKERS = 2 if os.name == 'nt' else min(8, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()

# Download the dataset using kagglehub
dataset_path = kagglehub.dataset_download('dondirecto/dr-training')
print(f"Dataset path: {dataset_path}")

# Set base paths to point to the local workspace folder
BASE_DIR = os.path.join(dataset_path, 'cleaned_dataset')
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR = os.path.join(BASE_DIR, 'val')
TEST_DIR = os.path.join(BASE_DIR, 'test')

Using device: cuda
Using Colab cache for faster access to the 'dr-training' dataset.
Dataset path: /kaggle/input/dr-training


## 3. Configure Data Transforms and Loaders

In [3]:
from torch.utils.data import WeightedRandomSampler

import cv2
from PIL import Image

class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        img_np = np.array(img)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l_clahe = self.clahe.apply(l)
        lab_clahe = cv2.merge((l_clahe, a, b))
        img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
        return Image.fromarray(img_clahe)

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((300, 300)),
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val_test': transforms.Compose([
        transforms.Resize((300, 300)),
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {
    'train': datasets.ImageFolder(TRAIN_DIR, data_transforms['train']),
    'val': datasets.ImageFolder(VAL_DIR, data_transforms['val_test']),
    'test': datasets.ImageFolder(TEST_DIR, data_transforms['val_test'])
}

train_targets = image_datasets['train'].targets
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)

# Sampler weights for balancing minibatches - strict inverse to balance all classes
sampler_class_weights = 1.0 / torch.tensor(np.clip(class_counts, 1, None), dtype=torch.float)

# Add smoothing to avoid completely overloading minority classes to the point of extreme overfitting
sampler_class_weights = torch.pow(sampler_class_weights, 0.8)
sample_weights = sampler_class_weights[train_targets]

# Loss weights for stronger minority-class learning
class_weights_for_loss = 1.0 / torch.tensor(np.clip(class_counts, 1, None), dtype=torch.float)
class_weights_for_loss = torch.pow(class_weights_for_loss, 0.5) # Square root smoothing
class_weights_for_loss = class_weights_for_loss / class_weights_for_loss.mean()
class_weights_for_loss = class_weights_for_loss.to(device)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights), # keep total batches consistent
    replacement=True
)

train_loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'sampler': sampler,
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
}
val_test_loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'shuffle': False,
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
}
if NUM_WORKERS > 0:
    train_loader_kwargs['persistent_workers'] = True
    val_test_loader_kwargs['persistent_workers'] = True

dataloaders = {
    'train': DataLoader(image_datasets['train'], **train_loader_kwargs),
    'val': DataLoader(image_datasets['val'], **val_test_loader_kwargs),
    'test': DataLoader(image_datasets['test'], **val_test_loader_kwargs)
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"Classes: {class_names}")
print(f"Original Class Counts in Train: {class_counts}")
print(f"Sampler Class Weights: {sampler_class_weights}")
print(f"Loss Class Weights (normalized): {class_weights_for_loss}")
print(f"Total Dataset Sizes: {dataset_sizes}")

Classes: ['0', '1', '2', '3', '4']
Original Class Counts in Train: [6777 3712 2568 1321 1545]
Sampler Class Weights: tensor([0.0009, 0.0014, 0.0019, 0.0032, 0.0028])
Loss Class Weights (normalized): tensor([0.5999, 0.8105, 0.9745, 1.3587, 1.2564], device='cuda:0')
Total Dataset Sizes: {'train': 15923, 'val': 2090, 'test': 2426}


## 4. Define Evidential Deep Learning (Type 2) Loss

In [4]:
# EDL Type-2 helpers used by the training loop
def relu_evidence(y):
    return F.relu(y)


def softplus_evidence(y):
    return F.softplus(y)


def kl_divergence(alpha, num_classes, device):
    beta = torch.ones((1, num_classes), dtype=torch.float, device=device)
    sum_alpha = torch.sum(alpha, dim=1, keepdim=True)

    first_term = (
        torch.lgamma(sum_alpha)
        - torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
        + torch.sum(torch.lgamma(beta), dim=1, keepdim=True)
        - torch.lgamma(torch.sum(beta, dim=1, keepdim=True))
    )

    second_term = torch.sum(
        (alpha - beta) * (torch.digamma(alpha) - torch.digamma(sum_alpha)),
        dim=1,
        keepdim=True,
    )

    return first_term + second_term


def edl_type2_loss(
    logits,
    target,
    epoch_num,
    num_classes,
    annealing_step,
    device,
    class_weights=None,
    epsilon=0.1,
    ce_weight=0.45,
    kl_scale=0.03,
):
    evidence = softplus_evidence(logits)
    alpha = evidence + 1.0

    # One-hot target with optional label smoothing for better calibration.
    target_one_hot = F.one_hot(target, num_classes=num_classes).float()
    if epsilon > 0.0:
        target_one_hot = target_one_hot * (1.0 - epsilon) + (epsilon / num_classes)

    s = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / s

    mse = torch.sum((target_one_hot - probs) ** 2, dim=1, keepdim=True)
    var = torch.sum(alpha * (s - alpha) / (s * s * (s + 1.0)), dim=1, keepdim=True)
    edl_term = mse + var

    if class_weights is not None:
        sample_w = class_weights[target].unsqueeze(1)
        edl_term = edl_term * sample_w

    annealing_coef = min(1.0, epoch_num / float(annealing_step))
    kl_alpha = (alpha - 1.0) * (1.0 - F.one_hot(target, num_classes=num_classes).float()) + 1.0
    kl_term = annealing_coef * kl_divergence(kl_alpha, num_classes, device)

    if class_weights is not None:
        kl_term = kl_term * sample_w
        ce = F.cross_entropy(logits, target, weight=class_weights)
    else:
        ce = F.cross_entropy(logits, target)

    loss = ce_weight * ce + torch.mean(edl_term) + kl_scale * torch.mean(kl_term)
    return loss


print('EDL Type-2 loss helpers are ready.')

EDL Type-2 loss helpers are ready.


## 5. Initialize EfficientNet B3 Model

In [5]:
class EfficientNetB3EDL(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super(EfficientNetB3EDL, self).__init__()
        if pretrained:
            self.base_model = EfficientNet.from_pretrained('efficientnet-b3')
        else:
            self.base_model = EfficientNet.from_name('efficientnet-b3')

        in_features = self.base_model._fc.in_features

        # Stronger head improves class separation for adjacent DR grades.
        self.base_model._fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(p=0.35),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

model = EfficientNetB3EDL(num_classes=NUM_CLASSES, pretrained=True)
model = model.to(device)

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b3-5fb5a3c3.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b3-5fb5a3c3.pth


100%|██████████| 47.1M/47.1M [00:00<00:00, 331MB/s]


Loaded pretrained weights for efficientnet-b3


## 6. Implement Training and Validation Steps

In [ ]:
# Strong single-run baseline (kept for quick checks before full k-fold)
optimizer = optim.AdamW(
    [
        {'params': model.base_model._fc.parameters(), 'lr': 1e-3},
        {'params': [p for n, p in model.base_model.named_parameters() if not n.startswith('_fc')], 'lr': 3e-4},
    ],
    weight_decay=1e-2 # Increased to 1e-2 to help reduce validation loss
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

def train_model(model, dataloaders, optimizer, scheduler, num_epochs=NUM_EPOCHS):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = 1e10
    best_acc = 0.0
    best_f1 = 0.0
    epochs_no_improve = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': []
    }

    print("Freezing base model for initial warm-up...")
    for param in model.base_model.parameters():
        param.requires_grad = False
    for param in model.base_model._fc.parameters():
        param.requires_grad = True

    for epoch in range(1, num_epochs + 1):
        print(f'Epoch {epoch}/{num_epochs}')
        print('-' * 10)

        if epoch == 4:
            print("Unfreezing all layers for deep fine-tuning...")
            for param in model.base_model.parameters():
                param.requires_grad = True
            optimizer.param_groups[0]['lr'] = 1e-4  # head (was 2e-4)
            optimizer.param_groups[1]['lr'] = 1e-5  # backbone (was 5e-5)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []

            if phase not in dataloaders:
                print(f"{phase} dataloader not found. Skipping.")
                continue

            for inputs, labels in tqdm(dataloaders[phase], desc=phase):
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    evidence = softplus_evidence(outputs)
                    alpha = evidence + 1
                    _, preds = torch.max(alpha, 1)

                    # Pass class_weights=None to avoid double penalization over the WeightedRandomSampler
                    loss = edl_type2_loss(
                        outputs, labels, epoch, NUM_CLASSES, 25, device,
                        class_weights=None,
                        epsilon=0.1,
                        ce_weight=0.45,
                        kl_scale=0.03,
                    )

                    if phase == 'train':
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                all_preds.extend(preds.detach().cpu().numpy())
                all_labels.extend(labels.detach().cpu().numpy())

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            epoch_f1 = f1_score(all_labels, all_preds, average='macro')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            history[f'{phase}_f1'].append(epoch_f1)

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Macro-F1: {epoch_f1:.4f}")

            if phase == 'val':
                scheduler.step()
                if epoch_f1 > best_f1:
                    best_f1 = epoch_f1
                    best_acc = epoch_acc.item()
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), 'best_edl2_efficientnet.pth')
                    print(f" -> Best model saved! (Macro-F1: {best_f1:.4f})")
                else:
                    epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping triggered after {epochs_no_improve} epochs without improvement')
            break

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Macro-F1: {best_f1:.4f} | Best val Acc: {best_acc:.4f} | Best val Loss: {best_loss:.4f}')

    model.load_state_dict(best_model_wts)
    return model, history

## 7. K-Fold Cross-Validation (Stratified) and Fold Checkpoints
This section trains multiple folds on the training split and saves one best checkpoint per fold (`best_fold_1.pth`, ..., `best_fold_k.pth`).

Why this helps:
- Reduces dependency on one lucky/unlucky split
- Improves robustness for minority grades
- Produces diverse checkpoints for ensembling

In [ ]:
# Strong single-run baseline (kept for quick checks before full k-fold)
optimizer = optim.AdamW(
    [
        {'params': model.base_model._fc.parameters(), 'lr': 1e-3},
        {'params': [p for n, p in model.base_model.named_parameters() if not n.startswith('_fc')], 'lr': 3e-4},
    ],
    weight_decay=1e-2 # Increased to 1e-2 to help reduce validation loss
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

def train_model(model, dataloaders, optimizer, scheduler, num_epochs=NUM_EPOCHS):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = 1e10
    best_acc = 0.0
    best_f1 = 0.0
    best_score = 0.0
    epochs_no_improve = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': []
    }

    print("Freezing base model for initial warm-up...")
    for param in model.base_model.parameters():
        param.requires_grad = False
    for param in model.base_model._fc.parameters():
        param.requires_grad = True

    for epoch in range(1, num_epochs + 1):
        print(f'Epoch {epoch}/{num_epochs}')
        print('-' * 10)

        if epoch == 4:
            print("Unfreezing all layers for deep fine-tuning...")
            for param in model.base_model.parameters():
                param.requires_grad = True
            # Reduced LR slightly here to prevent destroying pretrained representation and reduce validation loss spikes
            optimizer.param_groups[0]['lr'] = 1e-4  # head (was 2e-4)
            optimizer.param_groups[1]['lr'] = 1e-5  # backbone (was 5e-5)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []

            if phase not in dataloaders:
                print(f"{phase} dataloader not found. Skipping.")
                continue

            for inputs, labels in tqdm(dataloaders[phase], desc=phase):
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    evidence = softplus_evidence(outputs)
                    alpha = evidence + 1
                    _, preds = torch.max(alpha, 1)

                    # We pass class_weights=None because the Dataloader already uses a WeightedRandomSampler. 
                    # Using both class_weights and WeightedRandomSampler can cause severe overpenalization and high val loss.
                    loss = edl_type2_loss(
                        outputs, labels, epoch, NUM_CLASSES, 25, device,
                        class_weights=None,
                        epsilon=0.1,
                        ce_weight=0.45,
                        kl_scale=0.03,
                    )

                    if phase == 'train':
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                        optimizer.step()

                running_loss += float(loss.detach()) * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                all_preds.extend(preds.detach().cpu().numpy())
                all_labels.extend(labels.detach().cpu().numpy())

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            epoch_f1 = f1_score(all_labels, all_preds, average='macro')
            per_class_recall = classification_report(
                all_labels,
                all_preds,
                output_dict=True,
                zero_division=0,
            )
            class2_recall = float(per_class_recall.get('2', {}).get('recall', 0.0))
            epoch_score = 0.7 * epoch_f1 + 0.3 * class2_recall

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            history[f'{phase}_f1'].append(epoch_f1)

            if phase == 'train':
                print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Macro-F1: {epoch_f1:.4f}")
            else:
                print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Macro-F1: {epoch_f1:.4f} Class-2 Recall: {class2_recall:.4f} Score: {epoch_score:.4f}")

            if phase == 'val':
                scheduler.step()
                if epoch_score > best_score:
                    best_score = epoch_score
                    best_f1 = epoch_f1
                    best_acc = epoch_acc.item()
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), 'best_edl2_efficientnet.pth')
                    print(f" -> Best model saved! (Score: {best_score:.4f} | Macro-F1: {best_f1:.4f} | Class-2 Recall: {class2_recall:.4f})")
                else:
                    epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping triggered after {epochs_no_improve} epochs without improvement')
            break

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Score: {best_score:.4f} | Best val Macro-F1: {best_f1:.4f} | Best val Acc: {best_acc:.4f} | Best val Loss: {best_loss:.4f}')

    model.load_state_dict(best_model_wts)
    return model, history

# Execute training so Colab shows progress output immediately.
print('Starting Cell 7 training run...')
trained_model, history = train_model(
    model, dataloaders, optimizer, scheduler, num_epochs=NUM_EPOCHS
)

# Create fold-style checkpoint names so Cell 8 can run even with a single training run.
torch.save(trained_model.state_dict(), 'best_fold_1_top1.pth')
torch.save(trained_model.state_dict(), 'best_fold_1_top2.pth')

best_macro_f1 = float(max(history['val_f1'])) if len(history['val_f1']) > 0 else 0.0
fold_results = [
    {
        'fold': 1,
        'top1_ckpt': 'best_fold_1_top1.pth',
        'top2_ckpt': 'best_fold_1_top2.pth',
        'macro_f1': best_macro_f1,
        'class2_recall': 0.0,
    }
]

print('Saved checkpoints: best_fold_1_top1.pth, best_fold_1_top2.pth')
print(f'Best validation Macro-F1 from this run: {best_macro_f1:.4f}')

Starting Cell 7 training run...
Freezing base model for initial warm-up...
Epoch 1/60
----------


train:   0%|          | 0/498 [00:00<?, ?it/s]

Train Loss: 1.1363 Acc: 0.4907 Macro-F1: 0.4853


val:   0%|          | 0/66 [00:00<?, ?it/s]

Val Loss: 1.0260 Acc: 0.5292 Macro-F1: 0.4977 Class-2 Recall: 0.6034 Score: 0.5294
 -> Best model saved! (Score: 0.5294 | Macro-F1: 0.4977 | Class-2 Recall: 0.6034)

Epoch 2/60
----------


...

Train Loss: 0.0978 Acc: 0.9779 Macro-F1: 0.9789


val:   0%|          | 0/66 [00:00<?, ?it/s]

Val Loss: 2.3211 Acc: 0.7646 Macro-F1: 0.6872 Class-2 Recall: 0.6580 Score: 0.6785

Epoch 29/60
----------


train:   0%|          | 0/498 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Checkpoint Ensembling on Test Set (Run Immediately After Cell 7)
Run this only after Cell 7 has produced `best_fold_*.pth` checkpoints.

Why this helps:
- Reduces variance from individual folds
- Usually improves macro-F1 and worst-class recall
- More stable than single-checkpoint predictions

In [ ]:
import glob


def load_fold_models(checkpoint_paths, device):
    model_entries = []
    for ckpt_path in checkpoint_paths:
        if not os.path.exists(ckpt_path):
            print(f'Skipping missing checkpoint: {ckpt_path}')
            continue

        m = EfficientNetB3EDL(num_classes=NUM_CLASSES, pretrained=False).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device))
        m.eval()

        fold_weight = 1.0
        if 'fold_results' in globals():
            try:
                matched = [fr for fr in fold_results if fr.get('top1_ckpt', '') == ckpt_path or fr.get('top2_ckpt', '') == ckpt_path]
                if len(matched) > 0:
                    fold_weight = float(max(0.7 * matched[0].get('macro_f1', 1e-6) + 0.3 * matched[0].get('class2_recall', 1e-6), 1e-6))
            except Exception:
                pass

        model_entries.append({
            'path': ckpt_path,
            'model': m,
            'weight': fold_weight,
            'temperature': 1.0,
        })
    return model_entries


def estimate_temperature_grid(model, dataloader_val, temp_grid=None):
    if temp_grid is None:
        temp_grid = [0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0]

    if dataloader_val is None:
        return 1.0

    model.eval()
    logits_all = []
    labels_all = []
    with torch.no_grad():
        for inputs, labels in dataloader_val:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            logits_all.append(logits)
            labels_all.append(labels)

    if len(logits_all) == 0:
        return 1.0

    logits_all = torch.cat(logits_all, dim=0)
    labels_all = torch.cat(labels_all, dim=0)

    best_temp = 1.0
    best_ce = float('inf')

    for t in temp_grid:
        scaled_logits = logits_all / t
        ce = F.cross_entropy(scaled_logits, labels_all).item()
        if ce < best_ce:
            best_ce = ce
            best_temp = t

    return float(best_temp)


def evaluate_ensemble(model_entries, dataloader):
    all_preds, all_labels, all_uncertainties = [], [], []

    weights = np.array([max(me['weight'], 1e-8) for me in model_entries], dtype=np.float32)
    weights = weights / weights.sum()

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Ensemble Testing'):
            inputs = inputs.to(device)
            labels = labels.to(device)

            prob_weighted_sum = None
            for me, w in zip(model_entries, weights):
                logits = me['model'](inputs)
                scaled_logits = logits / me['temperature']
                probs = torch.softmax(scaled_logits, dim=1)
                prob_weighted_sum = probs * w if prob_weighted_sum is None else prob_weighted_sum + (probs * w)

            confidence, preds = torch.max(prob_weighted_sum, 1)
            uncertainty = 1.0 - confidence

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_uncertainties.extend(uncertainty.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_uncertainties)


# Keep off by default to preserve flow order.
USE_ENSEMBLE = True

if USE_ENSEMBLE:
    top1_ckpts = sorted(glob.glob('best_fold_*_top1.pth'))
    top2_ckpts = sorted(glob.glob('best_fold_*_top2.pth'))
    candidate_ckpts = sorted({*top1_ckpts, *top2_ckpts})
    if len(candidate_ckpts) == 0:
        candidate_ckpts = sorted(glob.glob('best_fold_*.pth'))

    if len(candidate_ckpts) == 0:
        print('No fold checkpoints found. Run Cell 7 first.')
    else:
        print(f'Found fold checkpoints: {candidate_ckpts}')
        model_entries = load_fold_models(candidate_ckpts, device)

        if len(model_entries) == 0:
            print('No valid checkpoints could be loaded.')
        elif 'test' not in dataloaders:
            print('Test dataloader not found.')
        else:
            if 'val' in dataloaders:
                print('Calibrating per-fold temperature on validation set...')
                for me in model_entries:
                    me['temperature'] = estimate_temperature_grid(me['model'], dataloaders['val'])
                    print(
                        f"{os.path.basename(me['path'])}: "
                        f"weight={me['weight']:.4f}, temperature={me['temperature']:.2f}"
                    )
            else:
                print('Validation dataloader missing. Using default temperature=1.0 for all folds.')

            y_true_e, y_pred_e, u_e = evaluate_ensemble(model_entries, dataloaders['test'])

            acc_e = accuracy_score(y_true_e, y_pred_e)
            macro_f1_e = f1_score(y_true_e, y_pred_e, average='macro')
            weighted_f1_e = f1_score(y_true_e, y_pred_e, average='weighted')

            print(f"\nEnsemble Test Accuracy: {acc_e:.4f}")
            print(f"Ensemble Test Macro-F1: {macro_f1_e:.4f}")
            print(f"Ensemble Test Weighted-F1: {weighted_f1_e:.4f}")
            print('\nEnsemble Classification Report:')
            try:
                print(classification_report(y_true_e, y_pred_e, target_names=class_names, digits=4))
            except Exception:
                print(classification_report(y_true_e, y_pred_e, digits=4))

            cm_e = confusion_matrix(y_true_e, y_pred_e)
            plt.figure(figsize=(8, 6))
            sns.heatmap(cm_e, annot=True, fmt='d', cmap='Blues')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.title('Ensemble Test Confusion Matrix')
            plt.show()

            plt.figure(figsize=(8, 5))
            plt.hist(u_e[y_true_e == y_pred_e], bins=50, alpha=0.5, label='Correct Predictions')
            plt.hist(u_e[y_true_e != y_pred_e], bins=50, alpha=0.5, label='Incorrect Predictions')
            plt.xlabel('Epistemic Uncertainty')
            plt.ylabel('Frequency')
            plt.legend()
            plt.title('Ensemble Uncertainty Distribution')
            plt.show()

            # Simple abstention analysis: drop highest-uncertainty samples and report retained accuracy
            for q in [0.90, 0.95]:
                thr = np.quantile(u_e, q)
                keep = u_e <= thr
                kept_ratio = np.mean(keep)
                if np.sum(keep) > 0:
                    acc_kept = accuracy_score(y_true_e[keep], y_pred_e[keep])
                    macro_f1_kept = f1_score(y_true_e[keep], y_pred_e[keep], average='macro')
                    print(
                        f"Abstain top {(1.0 - q) * 100:.0f}% uncertainty -> "
                        f"Coverage: {kept_ratio:.3f}, Accuracy: {acc_kept:.4f}, Macro-F1: {macro_f1_kept:.4f}"
                    )
else:
    print('Ensemble is configured. Set USE_ENSEMBLE = True after Cell 7 finishes.')

## 9. Download Checkpoints (Final Step)
Run this after training/evaluation to download `kfold_checkpoints.zip`.

In [ ]:
import os
import glob
import zipfile

# Download helper for all fold checkpoints
fold_ckpts = sorted(glob.glob('best_fold_*.pth'))

if len(fold_ckpts) > 0:
    zip_path = 'kfold_checkpoints.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in fold_ckpts:
            zf.write(fp)
    model_path = zip_path
    print(f"Prepared archive with fold checkpoints: {fold_ckpts}")
else:
    model_path = None

if model_path is not None and os.path.exists(model_path):
    try:
        from google.colab import files
        print("Running in Google Colab. Initiating download...")
        files.download(model_path)
    except ImportError:
        from IPython.display import FileLink, display
        print("Not in Colab. Creating a universal download link...")
        display(FileLink(model_path, result_html_prefix="Click here to download: "))
else:
    print("No fold checkpoints found. Run Cell 7 first.")